In [1]:
import pandas as pd
import fastparquet as fp

Pre-processing Admission Files To Remove Newborn Data

In [ ]:
# Loading MIMIC CSV Admission files
# Loading the admissions CSV file to filter out rows containing "NEWBORN" in the DIAGNOSIS column
df_admission = pd.read_csv('MIMIC-III Dataset\MIMIC -III (10000 patients)\ADMISSIONS\ADMISSIONS_sorted.csv')

In [ ]:
#TODO: Removing all cases where the keyword "NEWBORN" is present in the DIAGNOSIS column, due to the fact that newborns have different
# medical conditions and treatments compared to other patients, which could skew the analysis if included. 
# By filtering out these cases, we can focus on the adult patient population and ensure that our analysis is more relevant and accurate for that group.

#? Defining the column and the keyword to remove
column_name = 'DIAGNOSIS' 
keyword = 'NEWBORN'  

#! Keeping only the rows where the keyword "Newborn" is NOT present
#? The ~ symbol is a "not" operator in pandas
df_filtered = df_admission[~df_admission[column_name].str.contains(keyword, na=False)]

# Save the result to a new admissions CSV file
df_filtered.to_csv('MIMIC-III Dataset\MIMIC -III (10000 patients)\ADMISSIONS\ADMISSIOMS_sorted_cleaned.csv', index=False)

print(f"Done! Rows containing '{keyword}' have been removed.") #Debugging statement

***MIMIC III Aggregated Dataset Manipulation & Cleaning***

In [ ]:
# Step 1: Load the combined dataset, which will be processed for further analysis
# -------------------------------------------------------------------------------

# Using the combined dataset, which will be processed for further analysis
file_path = "Final Datasets\Complete_MIMIC_Aggregated_Cleaned.parquet" # Adjust the path as needed. This is the Parquet file created in the previous step, which contains the aggregated and cleaned MIMIC dataset.
fdf = fp.ParquetFile(file_path)

# Loading the combined dataset into a DataFrame
df = fdf.to_pandas()


# Loading the final output path for the cleaned dataset
output_path = 'Final Datasets\\Complete_MIMIC_Aggregated_Cleaned.csv' # Adjust the path as needed. This is the CSV file that will be created after further processing the combined dataset.

**Row-Wise Filtering**

In [5]:
# Step 2: Removing rows with null values in HADM_ID columns
# ------------------------------------------------------------------------------

# Printing initial shape (Rows, Columns)
print(f"Initial shape: {df.shape}")

# Defining the columns where to check for nulls, any of these specific column's row is empty, the whole row will be dropped.
target_columns = ['HADM_ID']


# subset: the list of columns to check
# inplace=True: modifies the existing dataframe directly
df.dropna(subset=target_columns, inplace=True) # Drop rows with null values in those specific columns

#Printing the final shape
print(f"Final shape: {df.shape}")

df.to_csv(output_path, index=False)

print(f"\nSaved! Rows with nulls in {target_columns} have been removed.")

Initial shape: (10253, 26)
Final shape: (10253, 26)

Saved! Rows with nulls in ['HADM_ID'] have been removed.


**Coloumn-Wise Filtering**

In [6]:
# Step 3: Removing unnecessary columns
# ------------------------------------------------------------------------------

# Defining the columns to remove (as a list)
cols_to_remove = ['ADMITTIME', 'LANGUAGE',"RELIGION","DEATHTIME","SUBJECT_ID","ROW_ID","DISCHTIME","INSURANCE","EDREGTIME","EDOUTTIME","ADMISSION_LOCATION","TOTAL_LAB_EVENTS","TOTAL_PRESCRIPTIONS","TOTAL_DIAGNOSES","HOSPITAL_EXPIRE_FLAG","HAS_CHARTEVENTS_DATA", "DIAGNOSIS"]

# Dropping the columns
# axis=1 tells pandas to look for column names, not row numbers
df = df.drop(columns=cols_to_remove, errors='ignore')

df.to_csv(output_path, index=False)

print(f"Successfully saved cleaned file to: {output_path}")

Successfully saved cleaned file to: C:\Users\Dell\Desktop\Semester 6\Parallel & Distributed Computing\Project\Final Datasets\Complete_MIMIC_Aggregated_Cleaned.csv


In [ ]:
# Step 4: Collapsing all columns into one column
# ------------------------------------------------------------------------------ 


# Grabbing only the first 1000 rows for testing
# df_subset = df.head(1000).copy()

#Grabbing whole dataset for the final version
df_subset = df.copy()

# Defining the primary "Anchor" column if any, which will be kept as is (not combined)
id_col = 'HADM_ID'  

# list of all columns to collapse
# cols_to_combine = [col for col in df_subset.columns ] # If you want to combine all columns, including the ID column, use this line instead
cols_to_combine = [col for col in df_subset.columns if col != id_col]


# 5. Create the new "Collapsed" column
# We convert everything to string first (.astype(str)) to avoid errors
df_subset['Combined_Data'] = df_subset[cols_to_combine].apply(lambda x: ' | '.join(x.dropna().astype(str)), axis=1)

# 6. Drop the original individual columns and keep only the ID and the new one
df_final = df_subset[[id_col, 'Combined_Data']].copy()  # Keep only the ID and the new combined column
#df_final = df_subset['Combined_Data'].copy()  # Keep only the ID and the new combined column

# Save the final result to a new CSV file
Output_path = 'Final Datasets\AlgoTesting_Merged_Coloums_Dataset.csv' #Comment out this line if you want to save the file with the default name "Combined MIMIC-III Dataset.csv"
df_final.to_csv(Output_path, index=False)

print(f"Done! Columns {cols_to_combine} have been merged into 'Combined_Data'.")
print(df_final.head())

Done! Columns ['ADMISSION_TYPE', 'DISCHARGE_LOCATION', 'MARITAL_STATUS', 'ETHNICITY', 'DIAGNOSIS_NAMES', 'PRESCRIBED_DRUGS', 'MICROBIOLOGY_ORGANISMS', 'TOTAL_MV_PROCEDURES'] have been merged into 'Combined_Data'.
   HADM_ID                                      Combined_Data
0   145834  EMERGENCY | SNF | MARRIED | WHITE | Unspecifie...
1   185777  EMERGENCY | HOME WITH HOME IV PROVIDR | SINGLE...
2   107064  ELECTIVE | HOME HEALTH CARE | MARRIED | WHITE ...
3   150750  EMERGENCY | DEAD/EXPIRED | UNKNOWN/NOT SPECIFI...
4   194540  EMERGENCY | HOME HEALTH CARE | MARRIED | WHITE...
